In [5]:
import sys
from pathlib import Path
sys.path.append('..')

import pandas as pd
from sklearn.model_selection import train_test_split
from preprocessing import prepare_data

train = prepare_data('../data/train.csv')


In [2]:
train.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Relatives,IsChild,FareLog,IsAlone,Embarked_Q,Embarked_S,Title_Miss,Title_Mr,Title_Mrs,Title_Rare
0,0,3,1,22.0,1,0,7.2500,1,0,2.110213,0,0,1,0,1,0,0
1,1,1,0,38.0,1,0,71.2833,1,0,4.280593,0,0,0,0,0,1,0
2,1,3,0,26.0,0,0,7.9250,0,0,2.188856,1,0,1,1,0,0,0
3,1,1,0,35.0,1,0,53.1000,1,0,3.990834,0,0,1,0,0,1,0
4,0,3,1,35.0,0,0,8.0500,0,0,2.202765,1,0,1,0,1,0,0


In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV

features = ['Pclass', 'Sex', 'Age', 'Fare', 'Relatives', 'Embarked_Q', 'Embarked_S', 'Title_Miss', 'Title_Mr', 'Title_Mrs']
X = train[features]

Y = train['Survived']

params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [4, 5, 6, 7, 8],
    'min_samples_split': [2, 5, 10]
}

grid = GridSearchCV(RandomForestClassifier(random_state=42), params, cv=5, scoring='accuracy')
grid.fit(X, Y)
print(grid.best_params_)
print(grid.best_score_)

model = grid.best_estimator_

for name, importance in zip(features, model.feature_importances_):
    print(f'{name}: {importance:.3f}')

{'max_depth': 8, 'min_samples_split': 5, 'n_estimators': 200}
0.8316678174628084
Pclass: 0.108
Sex: 0.163
Age: 0.127
Fare: 0.178
Relatives: 0.092
Embarked_Q: 0.009
Embarked_S: 0.017
Title_Miss: 0.040
Title_Mr: 0.221
Title_Mrs: 0.046


In [6]:
test = prepare_data('../data/test.csv')

test_predict = model.predict(test[features])

prediction = pd.DataFrame()

ids = pd.read_csv('../data/test.csv')['PassengerId']

prediction['PassengerId'] = ids
prediction['Survived'] = test_predict
Path('../predictions').mkdir(exist_ok=True)
prediction.to_csv('../predictions/prediction_rf.csv', index=False)